# FoodTrip — PhoBERT NER trên Kaggle GPU
Nhận diện tên quán (BUSINESS) và địa danh (LOC) trong caption video, **không phải AI tạo lịch trình**.

**Trước khi chạy** (cột phải → *Settings*):
1. *Accelerator*: **GPU T4 x2** (notebook chỉ dùng 1 GPU, giống lần train baseline).
2. *Internet*: **On** — cần tài khoản Kaggle đã xác minh số điện thoại.
3. *Add Input* → chọn dataset chứa **foodtrip-ner-colab-v3.zip** (bạn tải lên ở bước chuẩn bị).

Chạy lần lượt từng cell, hoặc *Save Version → Save & Run All* để chạy nền (tắt trình duyệt vẫn chạy).
Kết quả nằm ở tab *Output*: **foodtrip-ner-trained.zip** và **metrics.json**.
Không cần API key, không đăng nhập Hugging Face. Dữ liệu chỉ gồm caption công khai và code train.

## 1. Tìm bộ dữ liệu trong /kaggle/input
Kaggle tự giải nén file zip khi tạo dataset; cell này nhận cả thư mục đã giải nén lẫn file zip còn nguyên, và từ chối nếu nội dung không đúng bộ train cho phép.

In [ ]:
from pathlib import Path
import io, json, os, shutil, sys, subprocess, zipfile

INPUT, WORK, OUT = Path("/kaggle/input"), Path("/tmp/foodtrip-ner"), Path("/kaggle/working")
required = {"ner_preprocessing.py", "test_preprocessing.py", "check_segmentation.py", "check_tokenization.py", "train_ner.py", "requirements-colab.txt", "README.md",
            "data/train.jsonl", "data/val.jsonl", "data/test.jsonl", "data/val_template.jsonl", "data/test_template.jsonl",
            "data/labels.json", "data/quarantine.jsonl", "data/dataset-report.json"}
allowed = required | {"tokenizer-cases.json", "check_onnx.py"}
LIMIT = 20_000_000

def listing(root):
    return {p.relative_to(root).as_posix(): p.stat().st_size for p in root.rglob("*") if p.is_file()}

source = None
for root in sorted({p.parent for p in INPUT.rglob("train_ner.py")}):
    files = listing(root)
    if required <= set(files) <= allowed and sum(files.values()) <= LIMIT:
        source = ("dir", root); break
if source is None:
    for archive_path in sorted(INPUT.rglob("*.zip")):
        with zipfile.ZipFile(archive_path) as archive:
            files = {i.filename: i.file_size for i in archive.infolist() if not i.is_dir()}
        if required <= set(files) <= allowed and sum(files.values()) <= LIMIT:
            source = ("zip", archive_path); break
if source is None:
    raise ValueError("Không thấy bộ dữ liệu hợp lệ trong /kaggle/input. Add Input → dataset chứa foodtrip-ner-colab-v3.zip "
                     "(nội dung zip phải là các file train, không lồng thêm thư mục).")

if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
if source[0] == "dir":
    for name in files:
        (WORK / name).parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source[1] / name, WORK / name)
else:
    with zipfile.ZipFile(source[1]) as archive:
        for name in files:
            (WORK / name).parent.mkdir(parents=True, exist_ok=True)
            (WORK / name).write_bytes(archive.read(name))
print("Bộ dữ liệu:", source[1])
report = json.loads((WORK / "data/dataset-report.json").read_text())
print(json.dumps({k: report.get(k) for k in ["version", "labels", "counts", "negative_captions", "caption_tokenizer"]}, ensure_ascii=False, indent=2))
if str(report.get("labels", "")).startswith("SILVER"):
    print("⚠ Nhãn SILVER (chưa người duyệt): chỉ chạy thử quy trình, không dùng số liệu để báo cáo chất lượng.")

## 2. Môi trường GPU riêng
Python 3.12 riêng bằng uv, cùng phiên bản thư viện như lần train baseline. Mất vài phút. Chờ dòng **GPU_SETUP_OK**.

In [ ]:
"""Kaggle GPU environment for the FoodTrip NER run (the Kaggle notebook pastes this file in).

Same pinned stack as setup_colab.py — Python 3.12 venv via uv, torch 2.6.0 cu124,
requirements-colab.txt — but everything heavy lives under /tmp: /kaggle/working
is kept as the notebook's output, and a multi-GB venv there would be saved too.
Needs the notebook's Internet setting ON (a phone-verified Kaggle account).
"""
import os
import socket
import subprocess
import sys
from pathlib import Path


def run(command, *, cwd=None, env=None):
    """Stream both output channels so installation failures are visible in the notebook."""
    process_env = dict(os.environ if env is None else env)
    process_env['PYTHONUNBUFFERED'] = '1'
    with subprocess.Popen(command, cwd=cwd, env=process_env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)


WORK = Path('/tmp/foodtrip-ner')
if not (WORK / 'requirements-colab.txt').is_file():
    raise RuntimeError('Chạy cell 1 (tìm bộ dữ liệu trong /kaggle/input) trước')
try:
    socket.create_connection(('pypi.org', 443), timeout=10).close()
except OSError:
    raise RuntimeError('Không có Internet. Bật Settings → Internet: On (tài khoản Kaggle cần xác minh số điện thoại), rồi chạy lại.')
run(['nvidia-smi'])
TOOLS = Path('/tmp/foodtrip-uv-tools')
run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
     '--target', str(TOOLS), '--upgrade', 'uv==0.9.5'])
uv_env = dict(os.environ)
uv_env['PYTHONPATH'] = str(TOOLS)
uv_env['UV_PYTHON_INSTALL_DIR'] = '/tmp/foodtrip-uv-python'
UV = [sys.executable, '-m', 'uv']
run(UV + ['python', 'install', '3.12'], env=uv_env)
ENV = Path('/tmp/foodtrip-ner-py312')
PY = str(ENV / 'bin/python')
if not Path(PY).exists():
    run(UV + ['venv', '--python', '3.12', '--seed', str(ENV)], env=uv_env)
run([PY, '-c', 'import sys; print(sys.version); assert sys.version_info[:2] == (3, 12)'])
run([PY, '-m', 'pip', 'install', 'pip==25.0.1', 'setuptools==75.8.0', 'wheel', 'numpy==2.2.6'])
run([PY, '-m', 'pip', 'install', 'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124'])
run([PY, '-m', 'pip', 'install', '--no-build-isolation', '-r', str(WORK / 'requirements-colab.txt')])
run([PY, '-m', 'pip', 'check'])
# One GPU, like the baseline run: with "T4 x2" the trainer would otherwise split each
# batch over both cards and the effective batch size would differ from the baseline.
GPU_ENV = dict(os.environ, CUDA_VISIBLE_DEVICES='0')
run([PY, '-c', "import torch; print(torch.__version__); assert torch.cuda.is_available(), 'Chưa có GPU'; print(torch.cuda.get_device_name(0))"], env=GPU_ENV)
print('GPU_SETUP_OK — tiếp tục mục 3. Kiểm tra nhãn/tách từ và smoke test', flush=True)

## 3. Kiểm tra nhãn/tách từ và smoke test
Smoke chỉ hai bước, **không phải train xong**. Có lỗi thì dừng, không chạy cell tiếp theo.

In [ ]:
run([PY, str(WORK / "test_preprocessing.py")], cwd=WORK)
run([PY, str(WORK / "check_segmentation.py"), "--data", "data"], cwd=WORK)
run([PY, str(WORK / "check_tokenization.py"), "--data", "data"], cwd=WORK)
run([PY, str(WORK / "train_ner.py"), "--data", "data", "--output", "runs/smoke", "--smoke"], cwd=WORK, env=GPU_ENV)

## 4. Full fine-tune
Batch 4 × accumulation 4; FP16 và gradient checkpointing; tối đa 6 epochs, early stopping theo validation thật. Trên T4 thường khoảng 30–60 phút.

In [ ]:
if not (WORK / "runs/smoke/smoke-passed.json").exists():
    raise RuntimeError("Smoke test chưa qua")
command = [PY, str(WORK / "train_ner.py"), "--data", "data", "--output", "runs/v3", "--export-onnx"]
RESUME = False  # Đổi True chỉ khi tiếp tục checkpoint của chính phiên này (phiên mới thì /tmp đã mất).
if RESUME:
    command.append("--resume")
run(command, cwd=WORK, env=GPU_ENV)

## 5. Kiểm tra ONNX khớp PyTorch
Nếu bước xuất ONNX chưa tạo được báo cáo đối chiếu, chạy lại riêng phần so sánh (không train lại).

In [ ]:
RESULT = WORK / "runs/v3"
if not (RESULT / "onnx/export-check.json").exists() and (WORK / "check_onnx.py").exists():
    run([PY, str(WORK / "check_onnx.py"), "--bundle", str(RESULT), "--write-report"], cwd=WORK, env=GPU_ENV)
check = RESULT / "onnx/export-check.json"
print("ONNX:", json.loads(check.read_text()).get("passed") if check.exists() else "chưa có báo cáo đối chiếu")

## 6. Kết quả
Báo riêng test caption thật và test câu mẫu. `negative_captions.rate` = tỷ lệ caption không có quán mà model vẫn bắt ra một quán (mục tiêu ≤ 10%). Không điều chỉnh model theo test.

In [ ]:
metrics = json.loads((RESULT / "metrics.json").read_text())
for split, result in metrics.items():
    entities = result.get("entities", {})
    line = {k: round(entities[k]["f1-score"], 4) for k in ["BUSINESS", "LOC", "micro avg"] if k in entities}
    print(split, result.get("sentences"), "câu", line, "caption âm:", result.get("negative_captions"))
shutil.copyfile(RESULT / "metrics.json", OUT / "metrics.json")

## 7. Đóng gói model
Ghi **foodtrip-ner-trained.zip** vào /kaggle/working (không kèm checkpoint/optimizer). Tải ở tab *Output* của notebook, hoặc của Version nếu chạy nền.

In [ ]:
bundle = OUT / "foodtrip-ner-trained.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for item in RESULT.rglob("*"):
        if item.is_file() and "checkpoints" not in item.relative_to(RESULT).parts:
            archive.write(item, item.relative_to(RESULT))
    archive.write(WORK / "ner_preprocessing.py", "ner_preprocessing.py")
    archive.write(WORK / "requirements-colab.txt", "requirements-colab.txt")
    archive.write(WORK / "data/dataset-report.json", "dataset-report.json")
print(bundle, round(bundle.stat().st_size / 1e6), "MB")

## Sau khi train
Giải nén foodtrip-ner-trained.zip vào `ml/models/<tên-run>/` trong dự án, rồi chạy (xem `ml/training/WORKFLOW-V3.md`):

```
node scripts/benchmark-detect-place.js --data ml/datasets/v3/test.jsonl --errors ml/models/<tên-run>/test-errors.json
```

Không nạp model vào Supabase Edge. Giữ bước tra cứu địa điểm thật và người dùng xác nhận.